In [1]:
from enum import unique
from itertools import groupby
from unicodedata import category

from vespa.package import Field, RankProfile, MatchPhaseRanking
%load_ext autoreload
%autoreload 2
import mycode.vap as vap

In [2]:
# Index about 10k docs
# have 10 categories
# Score random in the first phase
# match-phase-should take top1 group
# search for top-10 hits
#

In [3]:
app = vap.demo_application_package()

In [75]:
from vespa.package import Field, Diversity, MatchPhaseRanking, RankProfile

app.get_schema().document.add_fields(
    Field(name="category", type="string", indexing="attribute | summary", attribute=["fast-search"])
)
app.get_schema().add_rank_profile(
    RankProfile(
        name="matchphase",
        match_phase=MatchPhaseRanking(
            attribute="id",
            order="descending",
            max_hits=10
        ),
        first_phase="attribute(id)",
    ),
)
app.get_schema().add_rank_profile(
    RankProfile(
        name="matchphasediversity",
        match_phase=MatchPhaseRanking(
            attribute="id",
            order="descending",
            max_hits=10
        ),
        diversity=Diversity(
            attribute="category",
            min_groups=10,
        ),
        first_phase="random",
    ),
)

In [53]:
print(app.get_schema().schema_to_text)

schema doc {
    document doc {
        field id type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field category type string {
            indexing: attribute | summary
            attribute {
                fast-search
            }
        }
    }
    rank-profile fields inherits unranked {
        function id() {
            expression {
                attribute(id)
            }
        }
        first-phase {
            expression {
                0
            }
        }
        summary-features {
            id
        }
        match-features {
            id
        }
    }
    rank-profile matchphasediversity {
        diversity {
            attribute: category
            min-groups: 10
        }
        match-phase {
            attribute: id
            order: descending
            max-hits: 10
        }
        first-phase {
            expression {
                random
            }
  

In [76]:
from vespa.deployment import VespaDocker

# In case running colima on macos run the following
# !sudo ln -sf $HOME/.colima/default/docker.sock /var/run/docker.sock
vespa_docker = VespaDocker(
    container_image="vespaengine/vespa:8.650.16",
)
# Start a docker container and deploy the application package
client = vespa_docker.deploy(
    application_package=app,
)

Waiting for configuration server, 0/60 seconds...
Waiting for configuration server, 5/60 seconds...
Application is up!
Finished deployment.


In [72]:
vap.feed(client, docs=[{'id': index, 'category': int(index / 1000)} for index in range(10000)])

In [91]:
resp1 = client.query(body={
    'yql': 'select * from sources * where true',
    'hits': 400,
    'ranking.profile': 'matchphasediversity',
}).json
resp1['root']['children']

[{'fields': {'category': '0',
   'documentid': 'id:doc:doc::990',
   'sddocname': 'doc'},
  'id': 'id:doc:doc::990',
  'relevance': 0.994390441570431,
  'source': 'test_content'},
 {'fields': {'category': '3',
   'documentid': 'id:doc:doc::3998',
   'sddocname': 'doc'},
  'id': 'id:doc:doc::3998',
  'relevance': 0.992834297940135,
  'source': 'test_content'},
 {'fields': {'category': '0',
   'documentid': 'id:doc:doc::117',
   'sddocname': 'doc'},
  'id': 'id:doc:doc::117',
  'relevance': 0.9886319967918098,
  'source': 'test_content'},
 {'fields': {'category': '8',
   'documentid': 'id:doc:doc::8998',
   'sddocname': 'doc'},
  'id': 'id:doc:doc::8998',
  'relevance': 0.98593297181651,
  'source': 'test_content'},
 {'fields': {'category': '0',
   'documentid': 'id:doc:doc::112',
   'sddocname': 'doc'},
  'id': 'id:doc:doc::112',
  'relevance': 0.9833442424423992,
  'source': 'test_content'},
 {'fields': {'category': '2',
   'documentid': 'id:doc:doc::2989',
   'sddocname': 'doc'},
  'i

In [92]:
resp1

{'root': {'children': [{'fields': {'category': '0',
     'documentid': 'id:doc:doc::990',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::990',
    'relevance': 0.994390441570431,
    'source': 'test_content'},
   {'fields': {'category': '3',
     'documentid': 'id:doc:doc::3998',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::3998',
    'relevance': 0.992834297940135,
    'source': 'test_content'},
   {'fields': {'category': '0',
     'documentid': 'id:doc:doc::117',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::117',
    'relevance': 0.9886319967918098,
    'source': 'test_content'},
   {'fields': {'category': '8',
     'documentid': 'id:doc:doc::8998',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::8998',
    'relevance': 0.98593297181651,
    'source': 'test_content'},
   {'fields': {'category': '0',
     'documentid': 'id:doc:doc::112',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::112',
    'relevance': 0.9833442424423992,
    'source': 'test_content'},
   {'fields': {

In [97]:
from collections import Counter
Counter(list(map(lambda x: x['fields']['category'], resp1['root']['children'])))

Counter({'0': 140,
         '3': 12,
         '8': 12,
         '2': 12,
         '7': 12,
         '6': 12,
         '4': 12,
         '5': 12,
         '1': 12,
         '9': 12})

In [98]:
resp = client.query(body={
    'yql': 'select * from sources * where true',
    'hits': 400,
    'ranking.profile': 'matchphase',
    # 'ranking.matchPhase.diversity.minGroups': 0,
    # 'ranking.matchPhase.diversity.attribute': "id",
}).json

In [81]:
resp

{'root': {'children': [{'fields': {'category': '9',
     'documentid': 'id:doc:doc::9999',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::9999',
    'relevance': 9999.0,
    'source': 'test_content'},
   {'fields': {'category': '9',
     'documentid': 'id:doc:doc::9998',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::9998',
    'relevance': 9998.0,
    'source': 'test_content'},
   {'fields': {'category': '9',
     'documentid': 'id:doc:doc::9997',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::9997',
    'relevance': 9997.0,
    'source': 'test_content'},
   {'fields': {'category': '9',
     'documentid': 'id:doc:doc::9996',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::9996',
    'relevance': 9996.0,
    'source': 'test_content'},
   {'fields': {'category': '9',
     'documentid': 'id:doc:doc::9995',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::9995',
    'relevance': 9995.0,
    'source': 'test_content'},
   {'fields': {'category': '9',
     'documentid': 'id:doc:doc::9

In [100]:
Counter(list(map(lambda x: x['fields']['category'], resp['root']['children'])))

Counter({'9': 128, '0': 128})